In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

In [4]:
generator_llm= ChatGroq(model= "llama-3.1-8b-instant")
evaluator_llm = ChatGroq(model= "llama-3.1-8b-instant")
optimizer_llm= ChatGroq(model= "llama-3.1-8b-instant")

In [7]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [8]:
strutured_evaluator_llm= evaluator_llm.with_structured_output(TweetEvaluation)

In [5]:
class TweetState(TypedDict):
    topic :str
    tweet: str
    evaluation: Literal["approved","need_improvement"]
    feedback : str
    iteration: int
    max_iteration: int

In [6]:
def generate_tweet(state : TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    result= generator_llm.invoke(messages).content
    return { 'tweet': result }

In [9]:
def evaluate_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
    """)
    ]
    response= strutured_evaluator_llm.invoke(messages)
    return {'evaluation': response.evaluation, 'feedback': response.feedback}
    

In [10]:
def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    response=optimizer_llm.invoke(messages).content
    iteration= state['iteration']+1
    return {'tweet': response, 'iteration': iteration}

In [11]:
def route_evaluation(state: TweetState):
    if state['evaluation'] == 'approved' or state['iteration']>= state['max_iteration']:
        return 'approved'
    else: 
        return 'needs_improvement'

In [14]:
graph= StateGraph(TweetState)
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize',optimize_tweet)


graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',route_evaluation,{'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize','evaluate')

workflow=graph.compile()


In [17]:
initial_state={
    'topic': "Indian Railways",
    "iteration": 1,
    "max_iteration":5
}
workflow.invoke(initial_state)

{'topic': 'Indian Railways',
 'tweet': '"Just spent 5 hours on Indian Railways and I\'ve finally mastered the art of eating Maggi with one hand while apologizing to my seatmate with the other #IndianRailways #MaggiForLife"',
 'evaluation': 'approved',
 'feedback': "The tweet is well-written and showcases a relatable experience of traveling on Indian Railways. The humor is clever and the use of Maggi is a great cultural reference. However, the tweet could be more concise and punchy to make it even more effective. Overall, it's a funny and original tweet that has the potential to go viral.",
 'iteration': 1,
 'max_iteration': 5}